In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/cheminfo-models-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "09-01.py"

## QSAR 是什麼：分類與迴歸

首格準備執行環境，下一格寫出分析程式，末格執行並顯圖。練習可接在分析程式末端。第一次執行需連線取得 MoleculeNet 資料。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "book-moleculenet"))
cache.mkdir(parents=True, exist_ok=True)
options = dict(featurizer=dc.feat.MACCSKeysFingerprint(), splitter="scaffold",
               transformers=[], data_dir=str(cache), save_dir=str(cache), reload=False)
folder = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
folder.mkdir(exist_ok=True)

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error
rt, (train_r, valid_r, test_r), _ = dc.molnet.load_bace_regression(**options)
ct, (train_c, valid_c, test_c), _ = dc.molnet.load_bace_classification(**options)
assert rt == ["pIC50"] and ct == ["Class"]
for a, b in zip((train_r, valid_r, test_r), (train_c, valid_c, test_c)):
    assert np.array_equal(a.ids, b.ids) and np.array_equal(a.X, b.X)
    assert np.all(a.w > 0) and np.all(b.w > 0)

reg = DummyRegressor(strategy="median").fit(train_r.X, train_r.y[:, 0])
clf = DummyClassifier(strategy="most_frequent").fit(train_c.X, train_c.y[:, 0])
yp = reg.predict(valid_r.X)
cp = clf.predict(valid_c.X)
print("counts:", len(train_c), len(valid_c), len(test_c))
print("train median, validation MAE:", yp[0], mean_absolute_error(valid_r.y[:, 0], yp))
print("train classes:", np.unique(train_c.y, return_counts=True))
print("majority class, validation accuracy:", cp[0], accuracy_score(valid_c.y[:, 0], cp))
print("validation classes:", np.unique(valid_c.y, return_counts=True))

fig, ax = plt.subplots(figsize=(7, 4.2), layout="constrained")
edges = np.linspace(train_r.y.min() - .01, train_r.y.max() + .01, 22)
for value, color in [(0, "#39899a"), (1, "#b35a37")]:
    mask = train_c.y[:, 0] == value
    ax.hist(train_r.y[mask, 0], bins=edges, color=color, alpha=.75, label=f"Class {value}")
ax.axvline(yp[0], linestyle="--", color="#333333", label="Training median")
ax.set(xlabel="Measured pIC50", ylabel="Training molecules")
ax.legend(frameon=False)
fig.savefig(folder / "09-01-bace-tasks.png", dpi=150)

In [ ]:
from IPython.display import Image, display
subprocess.run([python, str(script)], check=True)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(out / "09-01-bace-tasks.png")))